# درس ۰۷ - الگوی طراحی برنامه‌ریزی

این دفترچه یادداشت الگوی طراحی **برنامه‌ریزی** را برای عوامل هوش مصنوعی با استفاده از چارچوب عامل مایکروسافت نشان می‌دهد.
شما یاد خواهید گرفت چگونه درخواست‌های سفر پیچیده را به زیروظایف ساختاریافته تقسیم کنید، آن‌ها را به عوامل متخصص محول کنید،
و طرح نهایی را اجرا کنید — همه این‌ها با استفاده از خروجی ساختاریافته مبتنی بر مدل‌های Pydantic.


## راه‌اندازی


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv -q

In [ ]:
import logging
logging.getLogger("agent_framework.foundry").setLevel(logging.ERROR)

import os, asyncio
import dotenv
from typing import Annotated
from pydantic import BaseModel
from agent_framework import tool
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## تجزیه وظیفه

تجزیه وظیفه هسته الگوی طراحی برنامه‌ریزی است. به جای اینکه از یک عامل واحد بخواهیم
یک درخواست پیچیده را به‌طور کامل مدیریت کند، مسئله را به **زیروظایف** کوچکتر و واضح تقسیم می‌کنیم.
هر زیروظیفه به یک عامل متخصص (مثلاً پروازها، هتل‌ها، فعالیت‌ها) با
اولویت‌ها و ترتیب وابستگی مشخص اختصاص می‌یابد.

این رویکرد مزایای متعددی دارد:
- **وضوح**: هر زیروظیفه یک مسئولیت واحد دارد.
- **موازی‌سازی**: زیروظایف مستقل می‌توانند همزمان اجرا شوند.
- **قابلیت اطمینان**: خطاها به زیروظایف منفرد محدود می‌شوند.
- **ردیابی بودجه**: هزینه‌ها برای هر زیروظیفه تخمین زده و جمع می‌شوند.


In [ ]:
class TravelSubTask(BaseModel):
    task_id: int
    description: str
    assigned_agent: str  # "flight_agent", "hotel_agent", "activity_agent"
    priority: str  # "high", "medium", "low"
    dependencies: list[int] = []


class TravelPlan(BaseModel):
    destination: str
    trip_duration_days: int
    subtasks: list[TravelSubTask]
    total_estimated_budget_usd: int
    notes: str

## ایجاد یک عامل برنامه‌ریزی با خروجی ساختاریافته

عامل برنامه‌ریزی به‌عنوان **هماهنگ‌کننده میز پذیرش** عمل می‌کند. با دریافت یک درخواست سفر در سطح کلان،
یک `TravelPlan` ساختاریافته تولید می‌کند — درخواست را به زیرکارها تجزیه می‌کند، اولویت‌ها را تنظیم می‌کند،
و وابستگی‌ها را شناسایی می‌کند تا یک مسئول پذیرایی یا لایه اجرایی بتواند کار را انجام دهد.


In [ ]:
planning_agent = client.as_agent(
    name="TravelPlanner",
    instructions="""You are a travel planning agent. When given a travel request:
1. Break it into specific subtasks (flights, hotels, activities, logistics)
2. Assign each subtask to the appropriate specialist agent
3. Set priorities and identify dependencies between tasks
4. Estimate the total budget""",
)

result = await planning_agent.run(
    "Plan a 7-day trip to Paris for a couple interested in art, cuisine, and history. Budget around $5000.",
    options={"response_format": TravelPlan}
)
if result:
    plan = result.value
    print(f"Destination: {plan.destination}")
    print(f"Duration: {plan.trip_duration_days} days")
    print(f"Budget: ${plan.total_estimated_budget_usd}")
    print(f"\nSubtasks:")
    for task in plan.subtasks:
        print(f"  [{task.priority}] {task.task_id}. {task.description} → {task.assigned_agent}")

## اجرای یک برنامه با ابزارهای تخصصی

وقتی مسئول پذیرش یک برنامه ساختارمند تولید کرد، **نماینده کنسیرج** آن را اجرا می‌کند.
هر ابزار تخصصی یک دسته از زیرکارها (پروازها، هتل‌ها، فعالیت‌ها) را مدیریت می‌کند. کنسیرج
به ترتیب وابستگی از طریق زیرکارهای برنامه عبور می‌کند و هر یک را به
ابزار مناسب ارسال می‌کند.


In [ ]:
@tool
def book_flight(
    destination: Annotated[str, "The destination city"],
    departure_date: Annotated[str, "Departure date (YYYY-MM-DD)"],
    return_date: Annotated[str, "Return date (YYYY-MM-DD)"],
) -> str:
    """Search and book flights for the trip."""
    return f"Flight booked to {destination}: {departure_date} → {return_date}, confirmation #FLT-{hash(destination) % 10000:04d}"


@tool
def reserve_hotel(
    city: Annotated[str, "The city for the hotel"],
    check_in: Annotated[str, "Check-in date (YYYY-MM-DD)"],
    check_out: Annotated[str, "Check-out date (YYYY-MM-DD)"],
    guests: Annotated[int, "Number of guests"],
) -> str:
    """Reserve a hotel room in the destination city."""
    return f"Hotel reserved in {city}: {check_in} to {check_out} for {guests} guests, confirmation #HTL-{hash(city) % 10000:04d}"


@tool
def book_activity(
    activity_name: Annotated[str, "Name of the activity or tour"],
    date: Annotated[str, "Date of the activity (YYYY-MM-DD)"],
    participants: Annotated[int, "Number of participants"],
) -> str:
    """Book a tour, museum visit, or other activity."""
    return f"Activity booked: {activity_name} on {date} for {participants} people, confirmation #ACT-{hash(activity_name) % 10000:04d}"


# Concierge agent that executes the plan using specialist tools
concierge_agent = client.as_agent(
    name="Concierge",
    instructions="""You are a travel concierge executing a structured travel plan.
Use the available tools to fulfil each subtask. Work through the subtasks in order,
respecting dependencies. Summarise the results when finished.""",
    tools=[book_flight, reserve_hotel, book_activity],
)

# Build a prompt from the plan produced above
if result.value:
    subtask_lines = "\n".join(
        f"- [{t.priority}] {t.task_id}. {t.description} (agent: {t.assigned_agent}, deps: {t.dependencies})"
        for t in plan.subtasks
    )
    execution_prompt = (
        f"Execute the following travel plan for {plan.destination} "
        f"({plan.trip_duration_days} days, ${plan.total_estimated_budget_usd} budget):\n"
        f"{subtask_lines}"
    )

    exec_response = await concierge_agent.run(execution_prompt)
    print(exec_response)

## خلاصه

در این درس شما الگوی طراحی **برنامه‌ریزی** برای عامل‌های هوش مصنوعی را یاد گرفتید:

1. **تفکیک وظایف** — یک عامل برنامه‌ریز میز پذیرش، درخواست سفر پیچیده را به
   زیرکارهای ساختارمند با استفاده از مدل‌های Pydantic تقسیم می‌کند و هر کدام را به یک عامل متخصص با اولویت‌ها
   و وابستگی‌ها اختصاص می‌دهد.
2. **خروجی ساختارمند** — با ارسال یک `response_format`، عامل یک شیء معتبرشده `TravelPlan`
   را به جای متن آزاد بازمی‌گرداند که باعث می‌شود پردازش‌های بعدی قابل اطمینان باشد.
3. **اجرای برنامه** — یک عامل پذیرش از طریق زیرکارها با استفاده از ابزارهای تخصصی
   (`book_flight`، `reserve_hotel`، `book_activity`) برنامه را اجرا کرده و نتایج را گزارش می‌دهد.

این الگو *چه کاری انجام شود* (برنامه‌ریزی) را از *چگونه انجام شود* (اجرا) جدا می‌کند و عامل‌ها را
ماژولارتر، قابل تست‌تر و آسان‌تر برای توسعه می‌کند.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**سلب مسئولیت**:
این سند با استفاده از سرویس ترجمه هوش مصنوعی [Co-op Translator](https://github.com/Azure/co-op-translator) ترجمه شده است. در حالی که ما در تلاش برای دقت هستیم، لطفاً توجه داشته باشید که ترجمه‌های خودکار ممکن است شامل خطاها یا نادرستی‌هایی باشند. سند اصلی به زبان مادری خود باید به عنوان منبع معتبر در نظر گرفته شود. برای اطلاعات حیاتی، ترجمه حرفه‌ای انسانی توصیه می‌شود. ما در قبال هرگونه سوء تفاهم یا برداشت نادرست ناشی از استفاده از این ترجمه مسئولیتی نداریم.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
